In [1]:
import torch

 IN MY OWN WORDS

|  x  |  w  | -> z , loss = z^2 

To propagate the error backwards, figure out what to adjust the weights by, 

loss.backward takes dL/dz (sensitivty of loss function to a change in z = 12),

multiplies it by the activation (technically transposed) to get the weight's gradient i.e, 12 * 3 = 36. 

What this is saying is that: 
    a) 12 (dL/dz) is the sensisitivy of the loss to a change in Z
    b) in turn the sensitivity of z to w weight was 3, because z = w * x, and deriving that wrt w is 1 * x (1 *3), then dz/dw is 12 * 3 = 36
    c) so the backpropagation is saying, nudging z down by 1 unit will nudge the loss down by 12 units, and because nudging w down by 1 unit will nudge z down by 3 units, so that means when i nudge w down 1, i nudge z down 3, which means loss is nudged down (12 * 3) or 36, so 1 nudge of w is 36 nudges of L, so the gradient (dL/dw) is 36

In [ ]:
w = torch.tensor(2.0, requires_grad = True)
x = torch.tensor(3.0)

#Forward pass, x is input, w is weight
z = w * x 

#Loss function is z^2, which means dL/dz = 2z, so gradient is 2(6) = 12. 
loss = z ** 2

#
loss.backward()

print(w.grad)

tensor(36.)


Here, we initialize a neural net using the Pytorch neural network module, creating a class that inherits from the module, we can specifify each layer with the number of features in and out, (e.g. 3x 2)specify a bias, do a forward pass that takes in an input and does the multiplication? (self.layer(x)) we then instatiate our class to do so. 

In [5]:
import torch 
import torch.nn as nn   

class TinyNet(nn.Module): 
    def __init__ (self):
        super().__init__()
        self.layer = nn.Linear(in_features=1, out_features=1,  bias=False)

    def forward(self, x):
        return self.layer(x)

model = TinyNet()

print("weight:", model.layer.weight)
print("requires_grad:", model.layer.weight.requires_grad)

weight: Parameter containing:
tensor([[0.5107]], requires_grad=True)
requires_grad: True


In [6]:
x = torch.tensor([[3.0]]) 
output = model(x)
print("output:", output)
print("output shape:", output.shape)

output: tensor([[1.5320]], grad_fn=<MmBackward0>)
output shape: torch.Size([1, 1])


here we define a target (1x 1 matrix that's [10]), then we take advantage of TinyNet's __call__ method calls the forward method on x (3) (method multiplies it by the [1x1] weight in self.layer, we define our loss function, and then call Tiny Net's backward to do backpropagation. our loss is calculated (1.5 - 10)^2, and the resulting weight gradient is calculated, dL/dW 

In [11]:
target = torch.tensor([[10.0]])

output = model(x)

loss = (output - target) ** 2
print("loss:", loss) 

loss.backward()
print("weight grad:", model.layer.weight.grad)

loss: tensor([[48.2162]], grad_fn=<PowBackward0>)
weight grad: tensor([[-41.6627]])


optimizer step...

looks like instead of our simple nudge of the weight by the gradient * learning rate, here we are using an optimizer SGD function that takes in model parameters (weights?) and the learning rate as argumnets , we peek into the weights in our one layer ([1x1], then we call optimizer's step method, which i think takes a step according to lr * the gradient we calculated in the previous step, we then check again to see how the weights change after the step, then we zero the accumulated gradients


In [13]:
model = TinyNet()                       # fresh weight, fresh graph
optimizer = optim.SGD(model.parameters(), lr=0.01)
x = torch.tensor([[3.0]])
target = torch.tensor([[10.0]])

optimizer.zero_grad()                   # <-- zero FIRST, kills accumulation
output = model(x)
loss = (output - target) ** 2
loss.backward()

g = model.layer.weight.grad.item()
w_before = model.layer.weight.item()
optimizer.step()
w_after = model.layer.weight.item()

print(f"gradient      : {g:.4f}")
print(f"lr * gradient : {0.01 * g:.4f}")
print(f"weight before : {w_before:.4f}")
print(f"weight after  : {w_after:.4f}")
print(f"actual change : {w_after - w_before:.4f}")

gradient      : -73.8628
lr * gradient : -0.7386
weight before : -0.7702
weight after  : -0.0315
actual change : 0.7386
